In [1]:
from Document import Document
from Corpus import Corpus
from SearchEngine import SearchEngine

import pandas as pd
from tqdm import tqdm

In [2]:
corpus_art=pd.read_csv('corpus_article.csv', sep=',')
print(corpus_art.head())


   id                                              titre           auteur  \
0   0  So You Want a Tree ID but Can’t Be Bothered To...       DanoPinyon   
1   1  New visitors; WELCOME! PLEASE SEE our posting ...        spiceydog   
2   2  Admiring this random tree they put in the midd...        Lijey_Cat   
3   3     ID this tree in my friend’s backyard- Virginia    Old-Till-3179   
4   4  I know this is super abstract, and I hope this...  lilaccheesecake   

                  date                                                url  \
0  2025-08-13 07:35:32                  https://imgur.com/gallery/gqEtagA   
1  2025-08-24 12:23:02  https://www.reddit.com/r/Tree/wiki/posting_gui...   
2  2026-11-02 02:40:31               https://i.redd.it/oncltt8zs3zh1.jpeg   
3  2026-11-02 14:02:49             https://www.reddit.com/gallery/1xlwl42   
4  2026-11-01 14:20:15               https://i.redd.it/erw9kfzw40zh1.jpeg   

                                               texte  
0  (This sea arch c

In [3]:
corp_champ = Corpus("Nature and tree")

# On utilise .iterrows() pour récupérer l'index et la ligne (row)
for index, row in corpus_art.iterrows():
    document = Document(
        str(row["titre"]), 
        str(row["auteur"]), 
        str(row["date"]), 
        str(row["url"]), 
        str(row["texte"])
    )
    corp_champ.add(document)


print(corp_champ.nom)
print(corp_champ.id_documents)




Nature and tree
2001


In [ ]:
#for i in tqdm(range(1), desc="initialisation"):
moteur = SearchEngine(corp_champ)

# Demande de recherche à l'utilisateur
requete_utilisateur = input("Que voulez-vous rechercher ? ")
nb_a_afficher = 7

#Récupération du DataFrame de résultats
resultats = moteur.search(requete_utilisateur, top_n=nb_a_afficher)

    # Affichage du DataFrame final
print("\n--- Résultats de la recherche ---")
print(resultats)


PARTIE 3

In [4]:
import ipywidgets as widgets
from IPython.display import display

import matplotlib.pyplot as plt
        

In [5]:
out = widgets.Output(layout={'border': '1px solid black'})
titre = widgets.Label(value='Moteur de recherche')

consigne = widgets.Label(value='Entrez des mots clés et selectionnez le nombre de documents que vous souhaitez :')

texte = widgets.Text(
    value='',     
    placeholder='Mots clés',
    description='rechercher',
    disabled=False   
)

nb_doc = widgets.IntSlider(     
    min=0,
    max=10,
    description="Nombre d’articles à extraire :"
   
)

button = widgets.Button(
    description='Rechercher',
    disabled=False,
    #button_style='',
    #tooltip='rechercher',
    icon='check' # (FontAwesome names without the `fa-` prefix)
)

resultats = widgets.Textarea(
    value='',
    placeholder="Documents",
    description='Résultats :',
    disabled=True,
    layout=widgets.Layout(height="100%", width="40%") 
)

concordancier = widgets.Textarea(
    value='',
    placeholder="contexte",
    description='Concordancier :',
    disabled=True,
    layout=widgets.Layout(height="100%", width="60%") 
)



In [6]:
out = widgets.Output(layout={'border': '1px solid black'})
out.clear_output()
arbre=SearchEngine(corp_champ)

col1 =widgets.HBox([ texte,nb_doc])
col2 = widgets.HBox([button,resultats,concordancier ])
display(widgets.VBox([titre,consigne,col1,col2,out]))

def on_button_clicked(b):
    out.clear_output()
    if texte.value!="":
        button.disabled = True
        results= arbre.search(texte.value, top_n=nb_doc.value)
        with out:        
            print(results) 
            #BONUS 8 Concordancier interractif
            #je n'ai pas réussi à faire un meilleur affichage, meme avec "GridBox"
            concordancier.value=corp_champ.concorde(texte.value, taille_contexte=30).to_string()
            
            #pour le graphique --> BONUS 7 (tentative) <--je n'arrivais pas à récupérer les éléments de la matrice tf idf
            #print(results.plot.barh())
            #plt.show()
            if not results.empty and 'score' in results.columns and 'titre' in results.columns:
                results.set_index('titre')['score'].plot(
                    kind='barh', 
                    figsize=(8, 4), 
                    color='royalblue',
                    title=f'Scores pour : "{texte.value}"'
                )
                plt.xlabel('Score de similarité')
                plt.ylabel('Documents')
                plt.show()
            
            
        
        button.disabled = False
        
        if not results.empty and 'titre' in results.columns:
            resultats.value = results.titre.to_string()    

        else:
            resultats.value = "Aucun résultat trouvé."
    else:
        resultats.value = "Entrez une requête."
        concordancier.value ="Aucun résultat trouvé"



        
button.on_click(on_button_clicked)






Construction de la matrice: 100%|██████████| 2001/2001 [00:22<00:00, 89.12it/s] 
